In [ ]:
#import all tools
import numpy as np
import pandas as pd
import scanpy as sc
import scrublet as scr
import scipy.io
import matplotlib.pyplot as plt
import os
import bbknn as bk
import scvelo as scv

In [ ]:
adatar = sc.read_10x_mtx(
    'data/',  # the directory with the `.mtx` file
    var_names='gene_symbols',                 
    cache=True)    

In [ ]:
adatar.var_names_make_unique()

In [ ]:
adatar.obs

In [ ]:
adata = sc.read_h5ad('data/all2_filter3_trim2.h5ad')

In [ ]:
#load custom annotations and add it to adata
anno = pd.read_csv("data/obs.csv")
anno1=anno['sample_batch']
anno1 = np.asarray(anno1)
adata.obs['sample_batch']=anno1

In [ ]:
adata
anno = pd.read_csv("data/corrAnno.csv")
anno1=anno['Cells']
anno1 = np.asarray(anno1)
adata.obs['Cells']=anno1
anno2=anno['cellsMI']
anno2 = np.asarray(anno2)
adata.obs['cellsMI']=anno2

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
bk.bbknn(adata)
sc.tl.umap(adata)
sc.settings.set_figure_params(dpi=300,fontsize=10)

In [ ]:
fibroblasts = adata[adata.obs['Cells']=='Fibroblasts']
results_file = 'data/fibroblasts.h5ad'
fibroblasts.write(results_file)

In [ ]:
fibroblasts
temp = [s.replace('-1','') for s in fibroblasts.obs_names]
temp = [s.replace('-2','') for s in temp]
temp2 = ["{}{}".format(a_, b_) for a_, b_ in zip(temp, fibroblasts.obs['sample_batch'])]
fibroblasts.obs_names = temp2

In [ ]:
len(adatar.obs_names[adatar.obs_names.isin(fibroblasts.obs_names)])

In [ ]:
len(adatar.obs_names[~adatar.obs_names.isin(fibroblasts.obs_names)])

In [ ]:
len(fibroblasts.obs_names[fibroblasts.obs_names.isin(adatar.obs_names)])

In [ ]:
fibroblasts.obs_names[~fibroblasts.obs_names.isin(adatar.obs_names)]

In [ ]:
adatar.obs_names

In [ ]:
fibroblastr = adatar[fibroblasts.obs_names]

In [ ]:
fibroblastr.shape

In [ ]:
fibroblasts.raw = fibroblastr.copy()
results_file = 'data/fibroblasts.h5ad'
fibroblasts.write(results_file)

In [ ]:
import anndata

In [ ]:
fibroblasts_raw = anndata.AnnData(X = fibroblasts.raw.X , obs = fibroblasts.obs, var = fibroblasts.raw.var, obsm = fibroblasts.obsm)
rfile = 'data/fibroblasts_raw.h5ad'
fibroblasts_raw.write(rfile)

In [ ]:
sc.tl.pca(fibroblasts, svd_solver='arpack')
bk.bbknn(fibroblasts)
sc.tl.umap(fibroblasts)

In [ ]:
fibroblasts_raw = sc.read_h5ad('data/fibroblasts_raw.h5ad')

In [ ]:
fibroblasts = sc.read_h5ad('data/fibroblasts.h5ad')

In [ ]:
fibroblasts.X.max()

In [ ]:
sc.tl.pca(fibroblasts_raw, svd_solver='arpack')
bk.bbknn(fibroblasts_raw)
sc.tl.umap(fibroblasts_raw)

In [ ]:
sc.tl.leiden(fibroblasts,resolution=0.9,n_iterations=-1)
sc.pl.umap(fibroblasts,color='leiden',legend_fontsize='large')

In [ ]:
sc.tl.rank_genes_groups(fibroblasts, 'leiden', method = 't-test', n_genes = 500, use_raw = True)
result = fibroblasts.uns['rank_genes_groups']
groups = result['names'].dtype.names
markers = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names','pvals_adj','logfoldchanges','scores']})
markers.head(10)